# Step 1: Install Necessary packages

In [11]:
%%capture
!pip3 install langchain-text-splitters
!pip3 install langchain-community

!pip3 install langchain-openai langchain-core transformers huggingface_hub
!pip3 install sentence_transformers chromadb python-dotenv torch torchvision
!pip3 install langchain-chroma langchain-huggingface langchain langchain-classic
!pip3 install wget

# Step 2: Import all required packages

# RAG: Preprocessing ( Load, Split, Embed, Store )

In [16]:
def warn(*args, **kwargs):
  pass
import warnings
warnings.warn = warn
warnings.filterwarnings('ignore')

# Modern core chains replacing the classic/legacy ConversationalRetrievalChain
from langchain_classic.chains import create_history_aware_retriever, create_retrieval_chain
from langchain_classic.chains.combine_documents import create_stuff_documents_chain

# Rest of your foundational imports
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder
from langchain_core.runnables.history import RunnableWithMessageHistory
from langchain_community.document_loaders import TextLoader
from langchain_text_splitters import CharacterTextSplitter
from langchain_chroma import Chroma
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_openai import ChatOpenAI
from google.colab import userdata

import wget
import os

api_key = userdata.get('OPENAI_API_KEY')
os.environ['OPENAI_API_KEY'] = api_key


# Step 3: Load the document

filename = 'companyPolicies.txt'
url = 'https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/6JDbUb_L3egv_eOkouY71A.txt'

# Use wget to download the file
wget.download(url, out=filename)

with open(filename, 'r') as f:
  text = f.read()

# Step 4: Split the documents into chunks

loader = TextLoader(filename)
documents = loader.load()
text_split = CharacterTextSplitter(chunk_size=1000, chunk_overlap=0, separator='\n')
texts = text_split.split_documents(documents)
#print(len(texts))

# Step 5: Embedding and Storing into Chroma
#converting chunks to indexing (where unique number to identify each chunk) into chroma for easy search or retrival in future.

embedding = HuggingFaceEmbeddings(model_name='sentence-transformers/all-MiniLM-L6-v2')
vector_store = Chroma.from_documents(texts, embedding)
print('Documents store in Chroma vector db')

# Step 5: LLM Model

model = ChatOpenAI(
    model='gpt-4o-mini',
    temperature=0.5,
    max_tokens=256,
)

# Step 6: Integrating with LangChain

# 1. Initialize the LLM and your vector store retriever
retriever = vector_store.as_retriever(search_kwargs={"k": 3})

# 2. Define your system prompt
system_prompt = (
    "Use the given context to answer the question. "
    "If you don't know the answer, say you don't know.\n\n"
    "Context: {context}"
)
prompt = ChatPromptTemplate.from_messages([
    ("system", system_prompt),
    ("human", "{input}"),
])

# 3. Create the document combination chain
combine_docs_chain = create_stuff_documents_chain(model, prompt)

# 4. Create the final retrieval chain
retrieval_chain = create_retrieval_chain(retriever, combine_docs_chain)

# 5. Invoke the pipeline
response = retrieval_chain.invoke({"input": "What is mobile Policy?"})
#print(response["answer"])


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Documents store in Chroma vector db
